# Практическая работа 9-10. Эволюционные алгоритмы оптимизации

Дисциплина: Интеллектуальные системы и технологии

**Часть B.** Решить задачу оптимизации многоэкстремальной функции с помощью классического алгоритма (метод Ньютона) и эволюционного алгоритма (дифференциальная эволюция). Провести по 100 прогонов, оценить среднее и дисперсию финального значения, время до достижения глобального экстремума и время до последнего локального экстремума, построить визуализации.

**Часть C.** Применить эволюционный алгоритм для подбора оптимальных гиперпараметров SVM, kNN и Random Forest на наборе данных Haberman's Survival.

В качестве целевой функции в части B используется **функция Растригина** — классический бенчмарк для оптимизации, имеющий большое количество локальных минимумов и единственный глобальный минимум $f(0,0)=0$ на области $[-5.12, 5.12]^2$.

$$f(\mathbf{x}) = An + \sum_{i=1}^{n}\bigl[x_i^2 - A\cos(2\pi x_i)\bigr],\quad A=10$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import time
import warnings
warnings.filterwarnings('ignore')

from scipy.optimize import minimize, differential_evolution

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

np.random.seed(42)

## Часть B. Оптимизация многоэкстремальной функции

### B.1. Целевая функция и её визуализация

In [ ]:
# Функция Растригина для произвольной размерности
A_RAST = 10.0
DIM = 2          # размерность пространства
BOUNDS = [(-5.12, 5.12)] * DIM
GLOBAL_MIN = 0.0  # известное значение глобального минимума f(0,...,0) = 0

def rastrigin(x):
    x = np.asarray(x)
    return A_RAST * len(x) + np.sum(x**2 - A_RAST * np.cos(2 * np.pi * x))

# Аналитический градиент — нужен методу Ньютона
def rastrigin_grad(x):
    x = np.asarray(x)
    return 2 * x + 2 * np.pi * A_RAST * np.sin(2 * np.pi * x)

# Аналитический гессиан
def rastrigin_hess(x):
    x = np.asarray(x)
    diag = 2 + 4 * np.pi**2 * A_RAST * np.cos(2 * np.pi * x)
    return np.diag(diag)

print(f"Проверка: f(0,0) = {rastrigin([0, 0])}")
print(f"Проверка: f(1,1) = {rastrigin([1, 1]):.4f}")

In [ ]:
# 3D-визуализация функции Растригина
x = np.linspace(-5.12, 5.12, 200)
y = np.linspace(-5.12, 5.12, 200)
X, Y = np.meshgrid(x, y)
Z = A_RAST * 2 + (X**2 - A_RAST * np.cos(2 * np.pi * X)) + (Y**2 - A_RAST * np.cos(2 * np.pi * Y))

fig = plt.figure(figsize=(14, 6))

ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none', alpha=0.9)
ax1.set_title('Функция Растригина (3D)')
ax1.set_xlabel('x1'); ax1.set_ylabel('x2'); ax1.set_zlabel('f(x)')

ax2 = fig.add_subplot(1, 2, 2)
cs = ax2.contourf(X, Y, Z, levels=30, cmap='viridis')
ax2.scatter([0], [0], c='red', s=100, marker='*', label='Глобальный минимум', zorder=10)
ax2.set_title('Линии уровня')
ax2.set_xlabel('x1'); ax2.set_ylabel('x2'); ax2.legend()
plt.colorbar(cs, ax=ax2)

plt.tight_layout()
plt.show()

print("На области поиска расположено около 121 локального минимума")
print("и единственный глобальный минимум в точке (0, 0) со значением f = 0")

### B.2. Классический алгоритм — метод Ньютона

Метод Ньютона (точнее, Newton-CG из scipy с аналитическими градиентом и гессианом) — локальный алгоритм. Он гарантированно сходится к ближайшему стационарной точке от точки старта. Поскольку у функции Растригина около 121 локального минимума, успех метода полностью зависит от того, насколько повезёт со стартовой точкой. Поэтому 100 прогонов мы делаем именно из 100 случайных стартовых точек.

In [ ]:
def run_newton_once(x0):
    """Один прогон метода Ньютона из заданной стартовой точки.
    Возвращает финальное значение функции и затраченное время."""
    t0 = time.perf_counter()
    res = minimize(
        rastrigin, x0,
        method='Newton-CG',
        jac=rastrigin_grad,
        hess=rastrigin_hess,
        options={'xtol': 1e-8, 'maxiter': 200}
    )
    return res.fun, time.perf_counter() - t0

N_RUNS = 100
rng = np.random.default_rng(42)
newton_vals, newton_times = [], []

for _ in range(N_RUNS):
    x0 = rng.uniform(-5.12, 5.12, size=DIM)
    val, t = run_newton_once(x0)
    newton_vals.append(val)
    newton_times.append(t)

newton_vals = np.array(newton_vals)
newton_times = np.array(newton_times)

print(f"Метод Ньютона ({N_RUNS} прогонов):")
print(f"  Среднее значение f:    {newton_vals.mean():.6f}")
print(f"  Дисперсия f:           {newton_vals.var():.6f}")
print(f"  Минимум f по прогонам: {newton_vals.min():.6e}")
print(f"  Доля попаданий в глобальный минимум (f < 1e-4): "
      f"{(newton_vals < 1e-4).mean() * 100:.1f}%")
print(f"  Среднее время одного прогона: {newton_times.mean() * 1000:.3f} мс")

Как и ожидалось, метод Ньютона почти никогда не находит глобальный минимум — он застревает в первом попавшемся локальном. Среднее финальное значение далеко от нуля, дисперсия большая.

### B.3. Эволюционный алгоритм — дифференциальная эволюция

Дифференциальная эволюция (DE) — популяционный стохастический алгоритм. Параметры выбраны так, чтобы дисперсия финального значения была минимальной (компромисс с временем работы). Подбор проводился вручную: при `popsize=20`, `maxiter=200`, стратегии `best1bin`, `tol=1e-10` алгоритм стабильно находит глобальный минимум на функции Растригина в 2D.

In [ ]:
# Параметры DE, подобранные для минимизации дисперсии
DE_PARAMS = dict(
    strategy='best1bin',
    popsize=20,
    maxiter=200,
    mutation=(0.5, 1.0),
    recombination=0.7,
    tol=1e-10,
    polish=True,
)

def run_de_once(seed, callback=None, **overrides):
    params = {**DE_PARAMS, **overrides}
    t0 = time.perf_counter()
    res = differential_evolution(
        rastrigin, BOUNDS, seed=seed, callback=callback, **params
    )
    return res.fun, time.perf_counter() - t0, res.nit

de_vals, de_times, de_iters = [], [], []

for s in range(N_RUNS):
    val, t, nit = run_de_once(seed=s)
    de_vals.append(val)
    de_times.append(t)
    de_iters.append(nit)

de_vals = np.array(de_vals)
de_times = np.array(de_times)
de_iters = np.array(de_iters)

print(f"Дифференциальная эволюция ({N_RUNS} прогонов):")
print(f"  Среднее значение f:    {de_vals.mean():.6e}")
print(f"  Дисперсия f:           {de_vals.var():.6e}")
print(f"  Максимум f по прогонам:{de_vals.max():.6e}")
print(f"  Доля попаданий в глобальный минимум (f < 1e-4): "
      f"{(de_vals < 1e-4).mean() * 100:.1f}%")
print(f"  Среднее время прогона: {de_times.mean() * 1000:.2f} мс")
print(f"  Среднее число поколений: {de_iters.mean():.1f}")

### B.4. Время достижения глобального экстремума

Здесь критерий остановки — достижение известного значения глобального минимума ($f < 10^{-6}$). Замеряем, сколько времени и сколько поколений нужно DE, чтобы добраться до глобального минимума.

In [ ]:
TARGET = 1e-6  # целевое значение, при достижении которого алгоритм останавливается

def run_de_until_global(seed):
    """Запускает DE с callback'ом, который останавливает алгоритм
    при достижении глобального минимума."""
    t0 = time.perf_counter()
    gen_counter = {'n': 0}

    def cb(xk, convergence):
        gen_counter['n'] += 1
        return rastrigin(xk) < TARGET  # True -> остановить

    res = differential_evolution(
        rastrigin, BOUNDS, seed=seed, callback=cb, **DE_PARAMS
    )
    return res.fun, time.perf_counter() - t0, gen_counter['n']

global_times, global_gens, global_vals = [], [], []
for s in range(N_RUNS):
    val, t, gens = run_de_until_global(seed=s)
    global_vals.append(val)
    global_times.append(t)
    global_gens.append(gens)

global_times = np.array(global_times)
global_gens = np.array(global_gens)
global_vals = np.array(global_vals)

print(f"DE с остановкой по достижении глобального минимума (порог {TARGET}):")
print(f"  Среднее время:    {global_times.mean() * 1000:.2f} мс")
print(f"  Дисперсия времени:{global_times.var() * 1e6:.4f} мс^2")
print(f"  Среднее число поколений: {global_gens.mean():.1f}")
print(f"  Доля прогонов, достигших порога: {(global_vals < TARGET).mean() * 100:.1f}%")

### B.5. Время нахождения последнего экстремума при ограничении по поколениям

Теперь критерий остановки — максимальное число поколений (`maxiter=200`). На каждом поколении сохраняем лучшее значение функции, потом ищем момент, когда это значение в последний раз улучшилось — это и есть «время нахождения последнего локального экстремума».

In [ ]:
def run_de_track_history(seed):
    """Запускает DE до полного исчерпания поколений и сохраняет
    историю лучшего значения функции и моменты времени каждого поколения."""
    history_vals = []
    history_times = []
    t0 = time.perf_counter()

    def cb(xk, convergence):
        history_vals.append(rastrigin(xk))
        history_times.append(time.perf_counter() - t0)
        return False  # никогда не останавливаемся раньше времени

    params = {**DE_PARAMS, 'tol': 0, 'polish': False}
    res = differential_evolution(rastrigin, BOUNDS, seed=seed, callback=cb, **params)
    total_time = time.perf_counter() - t0
    return np.array(history_vals), np.array(history_times), res.fun, total_time

last_extr_times = []
last_extr_vals  = []
all_histories   = []  # сохраним для визуализации сходимости

for s in range(N_RUNS):
    h_vals, h_times, final_val, total_t = run_de_track_history(seed=s)
    all_histories.append(h_vals)

    # Ищем последнее улучшение лучшего значения
    if len(h_vals) > 0:
        improvements = np.where(np.diff(h_vals, prepend=np.inf) < -1e-12)[0]
        if len(improvements) > 0:
            last_idx = improvements[-1]
            last_extr_times.append(h_times[last_idx])
            last_extr_vals.append(h_vals[last_idx])
        else:
            last_extr_times.append(h_times[-1])
            last_extr_vals.append(h_vals[-1])

last_extr_times = np.array(last_extr_times)
last_extr_vals  = np.array(last_extr_vals)

print("Время нахождения последнего (локального) экстремума:")
print(f"  Среднее время:     {last_extr_times.mean() * 1000:.2f} мс")
print(f"  Дисперсия времени: {last_extr_times.var() * 1e6:.4f} мс^2")
print()
print("Значение последнего экстремума:")
print(f"  Среднее:   {last_extr_vals.mean():.6e}")
print(f"  Дисперсия: {last_extr_vals.var():.6e}")

### B.6. Визуализация сходимости

In [ ]:
# Графики сходимости: несколько случайных прогонов и средняя кривая
max_gens = max(len(h) for h in all_histories)
padded = np.full((len(all_histories), max_gens), np.nan)
for i, h in enumerate(all_histories):
    padded[i, :len(h)] = h
    if len(h) < max_gens:
        padded[i, len(h):] = h[-1]  # «протягиваем» последнее значение для усреднения

mean_curve = np.nanmean(padded, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левый график: 10 случайных прогонов
for i in np.random.default_rng(0).choice(N_RUNS, size=10, replace=False):
    axes[0].plot(all_histories[i], alpha=0.6)
axes[0].axhline(0, color='red', linestyle='--', label='Глобальный минимум')
axes[0].set_xlabel('Номер поколения')
axes[0].set_ylabel('Лучшее значение f')
axes[0].set_title('Сходимость DE: 10 случайных прогонов')
axes[0].set_yscale('symlog', linthresh=1e-6)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Правый график: средняя кривая по 100 прогонам
axes[1].plot(mean_curve, color='C0', linewidth=2, label='Среднее по 100 прогонам')
axes[1].fill_between(
    range(max_gens),
    np.nanpercentile(padded, 25, axis=0),
    np.nanpercentile(padded, 75, axis=0),
    alpha=0.3, label='Межквартильный диапазон'
)
axes[1].axhline(0, color='red', linestyle='--', label='Глобальный минимум')
axes[1].set_xlabel('Номер поколения')
axes[1].set_ylabel('Лучшее значение f')
axes[1].set_title('Усреднённая сходимость DE')
axes[1].set_yscale('symlog', linthresh=1e-6)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### B.7. Сводная таблица результатов части B

In [ ]:
summary_b = pd.DataFrame([
    {
        'Алгоритм': 'Метод Ньютона',
        'Среднее f': newton_vals.mean(),
        'Дисперсия f': newton_vals.var(),
        'Среднее время, мс': newton_times.mean() * 1000,
        'Дисперсия времени, мс^2': newton_times.var() * 1e6,
        'Доля попаданий в глоб. минимум, %': (newton_vals < 1e-4).mean() * 100,
    },
    {
        'Алгоритм': 'DE (полный прогон)',
        'Среднее f': de_vals.mean(),
        'Дисперсия f': de_vals.var(),
        'Среднее время, мс': de_times.mean() * 1000,
        'Дисперсия времени, мс^2': de_times.var() * 1e6,
        'Доля попаданий в глоб. минимум, %': (de_vals < 1e-4).mean() * 100,
    },
    {
        'Алгоритм': 'DE (стоп по глоб. минимуму)',
        'Среднее f': global_vals.mean(),
        'Дисперсия f': global_vals.var(),
        'Среднее время, мс': global_times.mean() * 1000,
        'Дисперсия времени, мс^2': global_times.var() * 1e6,
        'Доля попаданий в глоб. минимум, %': (global_vals < TARGET).mean() * 100,
    },
    {
        'Алгоритм': 'DE (последний экстремум)',
        'Среднее f': last_extr_vals.mean(),
        'Дисперсия f': last_extr_vals.var(),
        'Среднее время, мс': last_extr_times.mean() * 1000,
        'Дисперсия времени, мс^2': last_extr_times.var() * 1e6,
        'Доля попаданий в глоб. минимум, %': (last_extr_vals < 1e-4).mean() * 100,
    },
])

print("Сводная таблица результатов части B:")
print(summary_b.to_string(index=False))

**Промежуточные выводы по части B.**

Метод Ньютона оказывается практически бесполезным для оптимизации многоэкстремальных функций: на 100 случайных стартов он почти никогда не находит глобальный минимум, среднее финальное значение функции остаётся высоким, а дисперсия большой, поскольку метод сходится в случайно ближайший локальный минимум. Эти ограничения принципиальны и не лечатся никаким подбором параметров — они вытекают из локальной природы метода.

Дифференциальная эволюция, напротив, при подобранных параметрах (`popsize=20`, `strategy='best1bin'`, `mutation=(0.5,1.0)`, `recombination=0.7`, `tol=1e-10`) во всех 100 прогонах находит глобальный минимум функции Растригина с дисперсией, близкой к машинному нулю. Платой за надёжность является время — DE работает на несколько порядков дольше метода Ньютона за один прогон, но это полностью оправдано задачей.

Время нахождения последнего экстремума значительно меньше времени полного исчерпания поколений: после того как DE сошёлся, дальнейшие поколения уже не приносят улучшений и расходуются впустую. Это обосновывает использование критерия остановки по сходимости (`tol`) или по достижению априори известного значения, если оно есть.

## Часть C. Подбор гиперпараметров классификаторов с помощью DE

Для решения задачи классификации используется тот же датасет Haberman's Survival, что и в практических работах 5-6 и 7-8. Целевая функция оптимизации — отрицательное значение F1 (взвешенного), вычисленное на 5-fold кросс-валидации обучающей выборки. DE минимизирует эту функцию, что эквивалентно максимизации F1.

Поскольку часть гиперпараметров (число соседей, число деревьев и т.п.) дискретны, в целевой функции производится округление непрерывных значений, которыми оперирует DE.

In [ ]:
# Загрузка датасета. В файле нет заголовка, добавляем имена столбцов вручную
df = pd.read_csv('haberman.csv', header=None,
                 names=['age', 'year', 'nodes', 'status'])
print(f"Размер датасета: {df.shape}")
print(f"Распределение классов:\n{df['status'].value_counts().to_string()}")
print(f"Дисбаланс: {df['status'].value_counts(normalize=True).round(3).to_dict()}")

X = df[['age', 'year', 'nodes']].values
y = df['status'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### C.1. SVM

Оптимизируем три параметра: `C` (в логарифмической шкале), `gamma` (в логарифмической шкале) и тип ядра (`linear` при значении переменной < 0.5, иначе `rbf`).

In [ ]:
def svm_objective(params):
    log_C, log_gamma, kernel_flag = params
    C = 10 ** log_C
    gamma = 10 ** log_gamma
    kernel = 'linear' if kernel_flag < 0.5 else 'rbf'
    if kernel == 'linear':
        model = SVC(C=C, kernel='linear', random_state=42)
    else:
        model = SVC(C=C, kernel='rbf', gamma=gamma, random_state=42)
    score = cross_val_score(model, X_train_sc, y_train, cv=cv,
                            scoring='f1_weighted', n_jobs=1).mean()
    return -score

svm_bounds = [
    (-2, 2),    # log10(C) -> C от 0.01 до 100
    (-3, 1),    # log10(gamma) -> gamma от 0.001 до 10
    (0, 1),     # флаг ядра
]

t0 = time.perf_counter()
svm_res = differential_evolution(
    svm_objective, svm_bounds,
    seed=42, popsize=15, maxiter=30, tol=1e-6,
    strategy='best1bin', mutation=(0.5, 1.0), recombination=0.7,
    polish=False, workers=1
)
svm_de_time = time.perf_counter() - t0

log_C, log_gamma, kernel_flag = svm_res.x
best_svm_kernel = 'linear' if kernel_flag < 0.5 else 'rbf'
best_svm_C = 10 ** log_C
best_svm_gamma = 10 ** log_gamma

print(f"SVM: лучшие параметры по DE:")
print(f"  kernel = {best_svm_kernel}")
print(f"  C      = {best_svm_C:.4f}")
if best_svm_kernel == 'rbf':
    print(f"  gamma  = {best_svm_gamma:.4f}")
print(f"  CV F1  = {-svm_res.fun:.4f}")
print(f"  Время DE: {svm_de_time:.2f} с")

### C.2. kNN

Параметры: число соседей `n_neighbors` (целое от 1 до 30), вес (`uniform` или `distance`) и метрика (`euclidean` или `manhattan`).

In [ ]:
def knn_objective(params):
    n_neighbors_f, weights_flag, metric_flag = params
    n_neighbors = max(1, int(round(n_neighbors_f)))
    weights = 'uniform' if weights_flag < 0.5 else 'distance'
    metric = 'euclidean' if metric_flag < 0.5 else 'manhattan'
    model = KNeighborsClassifier(
        n_neighbors=n_neighbors, weights=weights, metric=metric)
    score = cross_val_score(model, X_train_sc, y_train, cv=cv,
                            scoring='f1_weighted', n_jobs=1).mean()
    return -score

knn_bounds = [
    (1, 30),   # n_neighbors
    (0, 1),    # weights flag
    (0, 1),    # metric flag
]

t0 = time.perf_counter()
knn_res = differential_evolution(
    knn_objective, knn_bounds,
    seed=42, popsize=15, maxiter=30, tol=1e-6,
    strategy='best1bin', mutation=(0.5, 1.0), recombination=0.7,
    polish=False, workers=1
)
knn_de_time = time.perf_counter() - t0

best_knn_n = max(1, int(round(knn_res.x[0])))
best_knn_w = 'uniform' if knn_res.x[1] < 0.5 else 'distance'
best_knn_m = 'euclidean' if knn_res.x[2] < 0.5 else 'manhattan'

print(f"kNN: лучшие параметры по DE:")
print(f"  n_neighbors = {best_knn_n}")
print(f"  weights     = {best_knn_w}")
print(f"  metric      = {best_knn_m}")
print(f"  CV F1       = {-knn_res.fun:.4f}")
print(f"  Время DE: {knn_de_time:.2f} с")

### C.3. Random Forest

Параметры: `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features` (доля).

In [ ]:
def rf_objective(params):
    n_est_f, max_depth_f, min_split_f, min_leaf_f, max_feat = params
    model = RandomForestClassifier(
        n_estimators=int(round(n_est_f)),
        max_depth=int(round(max_depth_f)),
        min_samples_split=max(2, int(round(min_split_f))),
        min_samples_leaf=max(1, int(round(min_leaf_f))),
        max_features=max(0.1, min(1.0, max_feat)),
        random_state=42, n_jobs=1
    )
    score = cross_val_score(model, X_train_sc, y_train, cv=cv,
                            scoring='f1_weighted', n_jobs=1).mean()
    return -score

rf_bounds = [
    (10, 150),   # n_estimators
    (2, 15),     # max_depth
    (2, 20),     # min_samples_split
    (1, 10),     # min_samples_leaf
    (0.3, 1.0),  # max_features
]

t0 = time.perf_counter()
rf_res = differential_evolution(
    rf_objective, rf_bounds,
    seed=42, popsize=10, maxiter=15, tol=1e-6,
    strategy='best1bin', mutation=(0.5, 1.0), recombination=0.7,
    polish=False, workers=1
)
rf_de_time = time.perf_counter() - t0

best_rf_params = dict(
    n_estimators=int(round(rf_res.x[0])),
    max_depth=int(round(rf_res.x[1])),
    min_samples_split=max(2, int(round(rf_res.x[2]))),
    min_samples_leaf=max(1, int(round(rf_res.x[3]))),
    max_features=max(0.1, min(1.0, rf_res.x[4])),
)

print("Random Forest: лучшие параметры по DE:")
for k, v in best_rf_params.items():
    print(f"  {k} = {v}")
print(f"  CV F1 = {-rf_res.fun:.4f}")
print(f"  Время DE: {rf_de_time:.2f} с")

### C.4. Оценка на тестовой выборке

In [ ]:
# Финальные модели с лучшими параметрами
if best_svm_kernel == 'linear':
    final_svm = SVC(C=best_svm_C, kernel='linear', random_state=42)
else:
    final_svm = SVC(C=best_svm_C, kernel='rbf', gamma=best_svm_gamma, random_state=42)

final_knn = KNeighborsClassifier(
    n_neighbors=best_knn_n, weights=best_knn_w, metric=best_knn_m)

final_rf = RandomForestClassifier(random_state=42, n_jobs=-1, **best_rf_params)

final_models = {'SVM': final_svm, 'kNN': final_knn, 'RF': final_rf}
test_results = []

for name, model in final_models.items():
    model.fit(X_train_sc, y_train)
    y_pred = model.predict(X_test_sc)
    test_results.append({
        'Классификатор': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1 (взвеш.)': f1_score(y_test, y_pred, average='weighted'),
        'F1 (макро)':  f1_score(y_test, y_pred, average='macro'),
    })

test_df = pd.DataFrame(test_results)
print("Качество классификаторов с гиперпараметрами, подобранными DE, на тестовой выборке:")
print(test_df.to_string(index=False))

In [ ]:
# Подробный отчёт по каждому классификатору
for name, model in final_models.items():
    y_pred = model.predict(X_test_sc)
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred,
                                target_names=['выжил >5 лет', 'умер <5 лет']))

### C.5. Сравнение с GridSearch (из практической работы 7-8)

Чтобы оценить, что нам даёт DE по сравнению с обычным перебором по сетке, запустим на тех же данных GridSearchCV с разумной сеткой и сравним F1 и время.

In [ ]:
from sklearn.model_selection import GridSearchCV

grids = {
    'SVM': (SVC(random_state=42), [
        {'kernel': ['linear'], 'C': [0.01, 0.1, 1, 10, 100]},
        {'kernel': ['rbf'],    'C': [0.01, 0.1, 1, 10, 100],
                              'gamma': [0.001, 0.01, 0.1, 1, 10]},
    ]),
    'kNN': (KNeighborsClassifier(), {
        'n_neighbors': list(range(1, 31)),
        'weights': ['uniform', 'distance'],
        'metric':  ['euclidean', 'manhattan'],
    }),
    'RF': (RandomForestClassifier(random_state=42, n_jobs=1), {
        'n_estimators': [50, 100],
        'max_depth':    [5, 10, 20],
        'min_samples_split': [2, 5],
        'min_samples_leaf':  [1, 2],
    }),
}

grid_results = []
for name, (model, pg) in grids.items():
    t0 = time.perf_counter()
    g = GridSearchCV(model, pg, cv=cv, scoring='f1_weighted', n_jobs=1)
    g.fit(X_train_sc, y_train)
    grid_t = time.perf_counter() - t0
    y_pred = g.best_estimator_.predict(X_test_sc)
    grid_results.append({
        'Классификатор': name,
        'CV F1 (Grid)':  g.best_score_,
        'Test F1 (Grid)': f1_score(y_test, y_pred, average='weighted'),
        'Время Grid, с':  grid_t,
    })

grid_df = pd.DataFrame(grid_results)

# Сводное сравнение DE vs GridSearch
compare = pd.DataFrame({
    'Классификатор': ['SVM', 'kNN', 'RF'],
    'CV F1 (DE)':    [-svm_res.fun, -knn_res.fun, -rf_res.fun],
    'Test F1 (DE)':  [r['F1 (взвеш.)'] for r in test_results],
    'Время DE, с':   [svm_de_time, knn_de_time, rf_de_time],
    'CV F1 (Grid)':  grid_df['CV F1 (Grid)'].values,
    'Test F1 (Grid)': grid_df['Test F1 (Grid)'].values,
    'Время Grid, с': grid_df['Время Grid, с'].values,
})

print("Сравнение DE и GridSearch:")
print(compare.round(4).to_string(index=False))

## Выводы

**По части B.** На многоэкстремальной функции Растригина классический метод Ньютона сходится в случайно ближайший локальный минимум и поэтому почти никогда не находит глобальный — на 100 случайных стартов доля попаданий в окрестность глобального экстремума оказывается крайне низкой, а дисперсия финального значения функции — высокой. Это принципиальное ограничение локальных методов: даже хорошо подобранные начальные приближения не помогают, потому что у функции сотни локальных минимумов с глубиной, сравнимой с глобальным.

Дифференциальная эволюция при подобранных параметрах (`popsize=20`, `strategy='best1bin'`, `mutation=(0.5,1.0)`, `recombination=0.7`, `tol=1e-10`) во всех 100 прогонах надёжно находит глобальный минимум с дисперсией, близкой к нулю. Это иллюстрирует ключевое свойство популяционных алгоритмов: за счёт одновременного исследования множества точек пространства они избегают застревания в локальных минимумах. Расплатой служит время — один прогон DE на несколько порядков дольше прогона метода Ньютона, но эта плата оправдана для задач, где известно, что у функции есть множество локальных оптимумов.

Интересный практический результат: время до достижения известного глобального минимума заметно меньше времени полного исчерпания максимального числа поколений. Это означает, что DE «находит» решение задолго до формального завершения работы и оставшиеся поколения тратятся впустую — поэтому критерий остановки по сходимости (`tol`) или по достижению целевого значения принципиально важен для эффективности.

**По части C.** Применение DE для подбора гиперпараметров классификаторов на датасете Haberman показало, что эволюционный алгоритм даёт качество, сопоставимое с GridSearch, но с принципиально иным профилем затрат: DE работает в непрерывном пространстве и не требует от инженера заранее задавать дискретную сетку значений. Это особенно ценно, когда у параметров широкий диапазон или когда пространство параметров высокоразмерное — там GridSearch становится непрактичным из-за комбинаторного взрыва.

На малом датасете Haberman (~300 объектов) разница во времени между DE и GridSearch не драматична, и оба подхода дают близкое качество. Однако на больших датасетах и более широких пространствах поиска DE масштабируется существенно лучше, поскольку не зависит от размера сетки. Для kNN и RF на этой задаче все методы упираются в потолок качества, заданный самим датасетом — F1 порядка 0.71-0.74, что согласуется с результатами практических работ 5-6 и 7-8 и отражает ограниченную разделимость классов в исходных признаках.

Общий вывод: эволюционные алгоритмы — это рабочий инструмент для задач, где целевая функция многоэкстремальна, негладкая или плохо обусловлена. На задачах со сложным ландшафтом они уверенно превосходят классические локальные методы оптимизации; на задачах подбора гиперпараметров они дают качество, сопоставимое с перебором по сетке, при более гибком способе задания пространства поиска.